In [16]:
import pandas as pd

input_occupation_titles = pd.read_csv("Task Pre-Filtering/Full Results/Evaluation/Top_10_Occupations_By_Rank_Diff_Aug.csv")
input_occupation_titles = input_occupation_titles["Title"].tolist()

full_dataset = pd.read_excel("Task Pre-Filtering/Dataset/TaskStatements.xlsx")
full_dataset = full_dataset[full_dataset["Title"].isin(input_occupation_titles)]
full_dataset = full_dataset[["Title", "Task"]].rename(columns={"Title": "Occupation"})

#group tasks by occupation and put them in a list
full_dataset = full_dataset.groupby("Occupation")["Task"]
full_dataset=full_dataset.apply(list)
full_dataset.to_csv("scenario_generation_context.csv", index=True)

In [17]:
import os
from dotenv import load_dotenv

# Load API key from environment variable or .env file
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found. Set it in your environment or .env file.")

os.environ["OPENAI_API_KEY"] = openai_api_key

In [18]:
with open("Task Pre-Filtering/Prompts/Scenario_Generation/scenario_generation_prompt.txt", "r", encoding="utf-8") as f:
    prompt = f.read()

In [19]:
N=0


occupation: Production, Planning, and Expediting Clerks
tasks: ['Distribute production schedules or work orders to departments.', 'Revise production schedules when required due to design changes, labor or material shortages, backlogs, or other interruptions, collaborating with management, marketing, sales, production, or engineering.', 'Review documents, such as production schedules, work orders, or staffing tables, to determine personnel or materials requirements or material priorities.', 'Arrange for delivery, assembly, or distribution of supplies or parts to expedite flow of materials and meet production schedules.', 'Confer with establishment personnel, vendors, or customers to coordinate production or shipping activities and to resolve complaints or eliminate delays.', 'Requisition and maintain inventories of materials or supplies necessary to meet production demands.', 'Confer with department supervisors or other personnel to assess progress and discuss needed changes.', 'Plan p

In [20]:
from plurals.agent import Agent
from plurals.deliberation import Ensemble, Moderator
import pandas as pd


df = pd.read_csv("scenario_generation_context.csv")
print(f"Total scenarios to process: {len(df)}")
for N in range(len(df)):
    occupation = df.iloc[N]['Occupation']
    tasks = df.iloc[N]['Task']
    context = f"""
occupation: {occupation}
tasks: {tasks}
"""
    print(f"Doing scenario {N} for: {context}")

    MODEL = "gpt-5-nano"
    task = prompt + "\n\n" + context
    agents = [
        Agent(
            model=MODEL,
            task=task,
            persona="""
You are an operations manager. Focus on timelines, resources, and delivery risks.
"""

        ),
        Agent(
            model=MODEL,
            task=task,
            persona="""
You are a frontline worker. Focus on practical problems, fatigue, mistakes, and poor procedures.
"""
        ),
        Agent(
            model=MODEL,
            task=task,
            persona="""
You are a safety risk specialist. Focus on hazards, compliance, and failure modes.
"""
        ),
        Agent(
            model=MODEL,
            task=task,
           persona="""
You are an external auditor. Focus on governance gaps, accountability, and overlooked risks.
"""
        ),
    ]

    moderator = Moderator(
        model=MODEL,
        combination_instructions = f"""
You are a moderator synthesizing risk scenarios from multiple agents.
their task is {task}
Your task is to combine all agent outputs into a single JSON object, with the format specified as in the prompt.

Combine insights from all agents, remove duplicates, and ensure 5 distinct scenarios, with a focus
on diversity and coverage of different risk severities.
"""
    )

    ensemble = Ensemble(
        agents=agents,
        moderator=moderator,
        task=task,
        cycles=2
    )
    print("Processing ensemble...")
    ensemble.process()
    print("Final Response from Moderator saved to risk_scenarios.json")
    with open("risk_scenarios_.json", "a", encoding="utf-8") as f:
        print(f"""
{{
"Occupation": "{occupation}",
"Tasks": "{tasks}",
"Generated_Scenarios":
{ensemble.final_response}
}},

""", file=f)


Total scenarios to process: 10
Doing scenario 0 for: 
occupation: Actors
tasks: ['Collaborate with other actors as part of an ensemble.', 'Portray and interpret roles, using speech, gestures, and body movements, to entertain, inform, or instruct radio, film, television, or live audiences.', 'Work closely with directors, other actors, and playwrights to find the interpretation most suited to the role.', 'Perform humorous and serious interpretations of emotions, actions, and situations, using body movements, facial expressions, and gestures.', 'Study and rehearse roles from scripts to interpret, learn and memorize lines, stunts, and cues as directed.', 'Learn about characters in scripts and their relationships to each other to develop role interpretations.', 'Attend auditions and casting calls to audition for roles.', 'Sing or dance during dramatic or comedic performances.', 'Work with other crew members responsible for lighting, costumes, make-up, and props.', 'Tell jokes, perform comic